# BRB-seq expression QC and PCA

This notebook compares deep non-UMI gene counts with deep UMI-deduplicated counts, maps barcodes to sample IDs, applies DESeq2's variance-stabilizing transform to the UMI counts, and makes separate gill and liver PCA plots. No differential-expression model is fitted here.

The compiled sample metadata are used for labels and annotations. `2024_DJR_196` is intentionally retained with unknown sex so its position can be inspected in the gill PCA.

In [1]:
library(DESeq2)
library(Matrix)
library(ggplot2)

# Edit these paths only if the repository/output location changes.
repo_root <- "/scratch/ztobias/Gacu_BC_txmics"
star_root <- file.path(repo_root, "outputs", "star")
metadata_file <- file.path(repo_root, "metadata", "derived", "sample_tracking.tsv")
barcode_file <- file.path(repo_root, "metadata", "brb_sample_barcodes.tsv")

tracking <- read.delim(metadata_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)
brb_barcodes <- read.delim(barcode_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)

tracking <- tracking[grepl("^(2024|2025)_DJR_", tracking$sample_id), ]
brb_barcodes <- brb_barcodes[grepl("^(2024|2025)_DJR_", brb_barcodes$sample_id), ]
head(tracking[, c("sample_id", "population", "ecotype", "field_sex", "true_sex")])

Loading required package: S4Vectors

Warning message:
“package ‘S4Vectors’ was built under R version 4.1.3”
Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Warning message:
“package ‘IRanges’ was built under R version 4.1.3”
Loading required packag

,sample_id,population,ecotype,field_sex,true_sex
,<chr>,<chr>,<chr>,<chr>,<chr>
1,2024_DJR_001,Priest,limnetic,M,M
2,2024_DJR_002,Priest,limnetic,F,F
3,2024_DJR_003,Priest,benthic,F,F
4,2024_DJR_004,Priest,limnetic,M,M
5,2024_DJR_005,Priest,limnetic,F,F
6,2024_DJR_006,Priest,benthic,M,M


## Read and map STAR's deep gene-count and UMI-count matrices

Each year/tissue run has its own STARsolo matrix. The 2025 gill and liver matrices already represent the independent tissue run plus the combined lane, because those FASTQs were concatenated before STAR. STAR was run with `NoDedup 1MM_Directional`, so the two matrices are `umiDedup-NoDedup.mtx` and `umiDedup-1MM_Directional.mtx`.

In [3]:
read_star_matrix <- function(matrix_path) {
  if (!file.exists(matrix_path)) stop("Missing STAR matrix: ", matrix_path)
  raw_dir <- dirname(matrix_path)
  barcode_path <- file.path(raw_dir, "barcodes.tsv")
  feature_path <- file.path(raw_dir, "features.tsv")
  x <- as.matrix(readMM(matrix_path))
  features <- read.delim(feature_path, header = FALSE, sep = "\t",
    stringsAsFactors = FALSE, check.names = FALSE)
  barcodes <- readLines(barcode_path)
  rownames(x) <- make.unique(features[[1]])
  colnames(x) <- barcodes
  storage.mode(x) <- "numeric"
  x
}

matrix_file <- function(tissue, year, mode = c("gene", "umi")) {
  mode <- match.arg(mode)
  prefix <- file.path(star_root, paste0(tissue, "_", year, "_deep_Solo.out"), "Gene", "raw")
  file.path(prefix, if (mode == "gene") "umiDedup-NoDedup.mtx" else "umiDedup-1MM_Directional.mtx")
}

map_matrix_to_samples <- function(x, tissue, year) {
  map <- brb_barcodes[brb_barcodes$tissue == tissue &
    brb_barcodes$year == as.integer(year), c("sample_id", "barcode")]
  map <- map[!duplicated(map$barcode), ]
  sample_names <- map$sample_id[match(colnames(x), map$barcode)]
  if (anyNA(sample_names)) {
    cat("Unmapped barcodes for ", tissue, " ", year, ":\n")
    print(colnames(x)[is.na(sample_names)])
  }
  keep <- !is.na(sample_names)
  x <- x[, keep, drop = FALSE]
  colnames(x) <- sample_names[keep]
  x
}

read_tissue_counts <- function(tissue, mode = c("gene", "umi")) {
  mode <- match.arg(mode)
  mats <- lapply(c("2024", "2025"), function(year) {
    x <- read_star_matrix(matrix_file(tissue, year, mode))
    map_matrix_to_samples(x, tissue, year)
  })
  genes <- Reduce(intersect, lapply(mats, rownames))
  mats <- lapply(mats, function(x) x[genes, , drop = FALSE])
  out <- do.call(cbind, mats)
  if (anyDuplicated(colnames(out))) stop("Duplicate sample IDs in ", tissue, " matrix")
  out
}

gill_gene_counts <- read_tissue_counts("gill", "gene")
liver_gene_counts <- read_tissue_counts("liver", "gene")
gill_umi_counts <- read_tissue_counts("gill", "umi")
liver_umi_counts <- read_tissue_counts("liver", "umi")

# The gene and UMI matrices should contain the same sample and gene sets.
stopifnot(identical(dim(gill_gene_counts), dim(gill_umi_counts)))
stopifnot(identical(dim(liver_gene_counts), dim(liver_umi_counts)))
dim(gill_gene_counts)
dim(liver_gene_counts)

Unmapped barcodes for  gill   2024 :
[1] "ACGATGCGACTACG" "ACTTAACCATACCA" "CCGCAATATGGCAA" "CTACTCAGTGTCGC"
[5] "CTATGCTAGCGAGC" "TATGCCATGCCTAC" "TCGACTGCATGCGA" "TCTTACAGCCAGCC"
[9] "TTACATCTAGTCCA"
Unmapped barcodes for  gill   2025 :
  [1] "AACAACGCCGGCCA" "AACAACTCGCACGG" "AACAAGGTGTCCAA" "AACCAGAGAATGGC"
  [5] "AACCGTGACATTAC" "AACCTGAGGCAAGA" "AACGGTACTGAAGA" "AACGTTAAGTCCGA"
  [9] "AAGAACGGCCTGGC" "AAGCATTCTGGCCG" "AAGCCGCACACAAG" "AAGCCGTGATTCCG"
 [13] "AAGGAAGTGCGCCA" "AAGGCGCTTCAACG" "AAGTGAATACCAAC" "AATCCTGGTACACG"
 [17] "AATGCCTTGTAAGG" "AATGGAACCGTGAG" "AATTATACCGCTCA" "ACAATACCGCCAAG"
 [21] "ACACACGAGCAGCG" "ACACATATGACAGC" "ACACGACAGCTCAC" "ACACTACTCGTGCA"
 [25] "ACAGATGCGTGCGC" "ACAGCAGTATAGCA" "ACAGCTGCTGTACA" "ACATAGTCATCTGA"
 [29] "ACATGGAGGAGAAC" "ACCGGCCAATAACA" "ACGATGCTCATCAC" "ACGGTTCAACATAA"
 [33] "ACGTACTAGAGCGA" "ACGTCAGCATCGAC" "ACTAGACGCACACA" "ACTCCATATTCGGC"
 [37] "ACTCTGCGCTCGAC" "ACTGAGTCTAGAGA" "ACTGGCCTAGTGGA" "ATACGCGACGCTCA"
 [41] "ATAGATACTGAGAG

[1] 28136   636

[1] 28136   639

In [4]:
make_coldata <- function(counts) {
  md <- tracking[match(colnames(counts), tracking$sample_id), , drop = FALSE]
  if (anyNA(md$sample_id)) stop("Count matrix sample missing from compiled metadata")
  rownames(md) <- md$sample_id
  md
}

gill_metadata <- make_coldata(gill_gene_counts)
liver_metadata <- make_coldata(liver_gene_counts)

# Unknown sex is retained as NA. Do not replace it before inspecting PCA.
gill_metadata[gill_metadata$sample_id == "2024_DJR_196",
  c("sample_id", "field_sex", "true_sex", "population", "ecotype")]

table(gill_metadata$year, useNA = "ifany")
table(liver_metadata$year, useNA = "ifany")

,sample_id,field_sex,true_sex,population,ecotype
,<chr>,<chr>,<chr>,<chr>,<chr>
2024_DJR_196,2024_DJR_196,?,,Boot,lake



2024 2025 
 375  261 


2024 2025 
 375  264 

## Gene-count versus UMI-count comparison

First compare total counts retained after UMI deduplication. A lower UMI/gene ratio indicates more PCR duplication. The population summaries are descriptive only; they show whether deduplication affects populations differently before choosing the expression matrix for PCA.

In [5]:
# Inspect the matrices before giving them to DESeq2.
count_diagnostics <- function(counts, label) {
  cat("\n--- ", label, " ---\n")
  cat("dimensions: ", nrow(counts), " genes x ", ncol(counts), " samples\n")
  cat("NA values: ", sum(is.na(counts)), "\n")
  cat("negative values: ", sum(counts < 0, na.rm = TRUE), "\n")
  cat("all-zero genes: ", sum(rowSums(counts) == 0), "\n")
  cat("all-zero samples: ", sum(colSums(counts) == 0), "\n")
  sample_summary <- data.frame(
    sample = colnames(counts),
    total_counts = colSums(counts),
    detected_genes = colSums(counts > 0),
    zero_genes = colSums(counts == 0)
  )
  print(summary(sample_summary$total_counts))
  print(summary(sample_summary$detected_genes))
  print(sample_summary[order(sample_summary$total_counts), ])
  invisible(sample_summary)
}

gill_gene_diagnostics <- count_diagnostics(gill_gene_counts, "gill gene")
liver_gene_diagnostics <- count_diagnostics(liver_gene_counts, "liver gene")
gill_umi_diagnostics <- count_diagnostics(gill_umi_counts, "gill UMI")
liver_umi_diagnostics <- count_diagnostics(liver_umi_counts, "liver UMI")


---  gill gene  ---
dimensions:  28136  genes x  636  samples
NA values:  0 
negative values:  0 
all-zero genes:  3806 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     52 2578656 2849312 2783620 3070169 6129952 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     48   15724   16192   16386   17652   19616 
                   sample total_counts detected_genes zero_genes
2025_DJR_419 2025_DJR_419           52             48      28088
2025_DJR_423 2025_DJR_423           59             52      28084
2025_DJR_415 2025_DJR_415           65             54      28082
2025_DJR_421 2025_DJR_421           68             59      28077
2025_DJR_246 2025_DJR_246           76             59      28077
2025_DJR_413 2025_DJR_413           79             65      28071
2025_DJR_416 2025_DJR_416           84             60      28076
2025_DJR_267 2025_DJR_267       223225          12225      15911
2025_DJR_286 2025_DJR_286       360418          13589      14547
2025_DJR_2


---  liver gene  ---
dimensions:  28136  genes x  639  samples
NA values:  0 
negative values:  0 
all-zero genes:  4139 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 115176 1813222 1992323 2167730 2607216 7050734 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   6103   12161   12703   12654   13260   16753 
                   sample total_counts detected_genes zero_genes
2024_DJR_104 2024_DJR_104       115176           6103      22033
2024_DJR_203 2024_DJR_203       128712           6305      21831
2024_DJR_094 2024_DJR_094       310024           7751      20385
2024_DJR_039 2024_DJR_039       383977           7966      20170
2024_DJR_105 2024_DJR_105       397768           9643      18493
2024_DJR_295 2024_DJR_295       505020          10724      17412
2024_DJR_085 2024_DJR_085       614961           9812      18324
2024_DJR_097 2024_DJR_097       649092          10668      17468
2024_DJR_009 2024_DJR_009       694703           9070      19066
2024_DJR_


---  gill UMI  ---
dimensions:  28136  genes x  636  samples
NA values:  0 
negative values:  0 
all-zero genes:  3806 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     52  797540  912545 1202620 1758084 4227987 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     48   15724   16192   16386   17652   19616 
                   sample total_counts detected_genes zero_genes
2025_DJR_419 2025_DJR_419           52             48      28088
2025_DJR_423 2025_DJR_423           59             52      28084
2025_DJR_415 2025_DJR_415           65             54      28082
2025_DJR_421 2025_DJR_421           68             59      28077
2025_DJR_246 2025_DJR_246           76             59      28077
2025_DJR_413 2025_DJR_413           78             65      28071
2025_DJR_416 2025_DJR_416           84             60      28076
2025_DJR_267 2025_DJR_267       154412          12225      15911
2025_DJR_286 2025_DJR_286       257644          13589      14547
2025_DJR_25


---  liver UMI  ---
dimensions:  28136  genes x  639  samples
NA values:  0 
negative values:  0 
all-zero genes:  4139 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  59729  988956 1142873 1088587 1226782 3103504 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   6103   12161   12703   12654   13260   16753 
                   sample total_counts detected_genes zero_genes
2024_DJR_104 2024_DJR_104        59729           6103      22033
2024_DJR_203 2024_DJR_203        63574           6305      21831
2024_DJR_094 2024_DJR_094       140468           7751      20385
2024_DJR_039 2024_DJR_039       175916           7966      20170
2024_DJR_105 2024_DJR_105       204832           9643      18493
2024_DJR_295 2024_DJR_295       256886          10724      17412
2024_DJR_085 2024_DJR_085       291846           9812      18324
2024_DJR_009 2024_DJR_009       299195           9070      19066
2024_DJR_097 2024_DJR_097       310168          10668      17468
2024_DJR_4

In [ ]:
summarize_count_ratio <- function(gene_counts, umi_counts, metadata, tissue) {
  stopifnot(identical(rownames(gene_counts), rownames(umi_counts)))
  stopifnot(identical(colnames(gene_counts), colnames(umi_counts)))
  sample_metadata <- metadata[match(colnames(gene_counts), metadata$sample_id), , drop = FALSE]
  total_gene <- colSums(gene_counts)
  total_umi <- colSums(umi_counts)
  data.frame(
    sample_id = colnames(gene_counts),
    tissue = tissue,
    year = sample_metadata$year,
    population = sample_metadata$population,
    ecotype = sample_metadata$ecotype,
    gene_counts = total_gene,
    umi_counts = total_umi,
    umi_to_gene_ratio = ifelse(total_gene > 0, total_umi / total_gene, NA_real_),
    stringsAsFactors = FALSE
  )
}

count_ratio_by_sample <- rbind(
  summarize_count_ratio(gill_gene_counts, gill_umi_counts, gill_metadata, "gill"),
  summarize_count_ratio(liver_gene_counts, liver_umi_counts, liver_metadata, "liver")
)

grouped <- split(seq_len(nrow(count_ratio_by_sample)),
  interaction(count_ratio_by_sample$tissue, count_ratio_by_sample$population,
    count_ratio_by_sample$ecotype, drop = TRUE))
count_ratio_by_population <- do.call(rbind, lapply(grouped, function(i) {
  x <- count_ratio_by_sample$umi_to_gene_ratio[i]
  data.frame(
    tissue = count_ratio_by_sample$tissue[i[1]],
    population = count_ratio_by_sample$population[i[1]],
    ecotype = count_ratio_by_sample$ecotype[i[1]],
    n = sum(!is.na(x)),
    median_umi_to_gene = median(x, na.rm = TRUE),
    mean_umi_to_gene = mean(x, na.rm = TRUE),
    sd_umi_to_gene = sd(x, na.rm = TRUE),
    stringsAsFactors = FALSE
  )
}))
row.names(count_ratio_by_population) <- NULL
count_ratio_by_sample[order(count_ratio_by_sample$tissue, count_ratio_by_sample$umi_to_gene_ratio), ]
count_ratio_by_population[order(count_ratio_by_population$tissue, count_ratio_by_population$population), ]

ggplot(count_ratio_by_sample, aes(x = population, y = umi_to_gene_ratio)) +
  geom_boxplot(outlier.size = 0.6, na.rm = TRUE) +
  facet_wrap(~ tissue, scales = "free_x") +
  coord_cartesian(ylim = c(0, 1)) +
  labs(title = "UMI counts retained relative to gene counts",
    x = "Population", y = "UMI counts / non-UMI gene counts") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

# Use UMI-deduplicated counts for the exploratory expression PCA.
make_vsd <- function(counts, metadata) {
  keep <- rowSums(counts >= 10) >= max(2, ceiling(ncol(counts) * 0.10))
  dds <- DESeqDataSetFromMatrix(
    countData = round(counts[keep, , drop = FALSE]),
    colData = metadata,
    design = ~ 1
  )
  list(
    dds = dds,
    vsd = vst(dds, blind = TRUE),
    genes_kept = sum(keep)
  )
}

gill_vsd <- make_vsd(gill_umi_counts, gill_metadata)
liver_vsd <- make_vsd(liver_umi_counts, liver_metadata)
c(gill = gill_vsd$genes_kept, liver = liver_vsd$genes_kept)

In [ ]:
make_pca_data <- function(vsd_obj, metadata) {
  pca <- prcomp(t(assay(vsd_obj)), center = TRUE, scale. = FALSE)
  variance <- 100 * pca$sdev^2 / sum(pca$sdev^2)
  out <- cbind(metadata,
    PC1 = pca$x[, 1], PC2 = pca$x[, 2])
  attr(out, "variance") <- variance
  out
}

gill_pca_data <- make_pca_data(gill_vsd$vsd, gill_metadata)
liver_pca_data <- make_pca_data(liver_vsd$vsd, liver_metadata)

plot_pca <- function(pca_data, title) {
  variance <- attr(pca_data, "variance")
  ggplot(pca_data, aes(PC1, PC2, color = ecotype, shape = true_sex)) +
    geom_point(size = 2.2, na.rm = FALSE) +
    geom_text(
      data = pca_data[pca_data$sample_id == "2024_DJR_196", , drop = FALSE],
      aes(label = sample_id), nudge_y = 0.02, na.rm = TRUE, show.legend = FALSE
    ) +
    facet_wrap(~ population) +
    labs(
      title = title,
      subtitle = "Exploratory UMI-count VST PCA; no DE model fitted",
      x = paste0("PC1 (", round(variance[1], 1), "%)"),
      y = paste0("PC2 (", round(variance[2], 1), "%)"),
      shape = "WGS/compiled sex"
    ) +
    theme_classic()
}

plot_pca(gill_pca_data, "Gill BRB-seq")
plot_pca(liver_pca_data, "Liver BRB-seq")

### Interpretation note

If `2024_DJR_196` clusters clearly with one sex in the gill PCA, that is useful evidence for follow-up, but it is not by itself a genetic-sex assignment. Population, ecotype, batch, sequencing depth, and sample-quality structure can also drive PCA separation.